In [94]:
import pandas as pd
import numpy as np
import json

In [95]:
transactions = pd.read_csv('../data/transactions_features.csv', parse_dates=['t_dat'])
customers = pd.read_csv('../data/customers_features.csv')
articles = pd.read_csv('../data/articles_features.csv')

In [97]:
customer_cols = [
    'customer_id', 'age', 'age_group', 
    'club_member_status', 'fashion_news_frequency'
]

article_cols = [
    'article_id', 'product_group_name', 'graphical_appearance_name',
    'colour_group_name', 'perceived_colour_value_name', 
    'index_group_name', 'index_name', 'section_name',
    'garment_group_name', 'item_type', 'price_segment'
]

master_exp_02 = transactions.merge(
    customers[customer_cols], 
    on='customer_id', 
    how='left'
)

master_exp_02 = master_exp_02.merge(
    articles[article_cols], 
    on='article_id', 
    how='left'
)

cat_cols = master_exp_02.select_dtypes(include=['object', 'category']).columns
master_exp_02[cat_cols] = master_exp_02[cat_cols].fillna('Unknown')

master_exp_02['age'] = master_exp_02['age'].fillna(master_exp_02['age'].median())

print(master_exp_02.shape)
print(list(master_exp_02.columns))

/var/folders/yq/ymvjb12x09x0xjvftjt24x2h0000gn/T/ipykernel_62100/1952026001.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = master_exp_02.select_dtypes(include=['object', 'category']).columns


(1054096, 19)
['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id', 'age', 'age_group', 'club_member_status', 'fashion_news_frequency', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'index_group_name', 'index_name', 'section_name', 'garment_group_name', 'item_type', 'price_segment']


In [ ]:
max_date = master_exp_02['t_dat'].max()
test_start = max_date - pd.Timedelta(days=7)
val_start = test_start - pd.Timedelta(days=7)

train_data = master_exp_02[master_exp_02['t_dat'] < val_start].copy()
val_data = master_exp_02[(master_exp_02['t_dat'] >= val_start) & (master_exp_02['t_dat'] < test_start)].copy()
test_data = master_exp_02[master_exp_02['t_dat'] >= test_start].copy()

In [ ]:
val_actual = val_data.groupby('customer_id')['article_id'].apply(list).to_dict()
test_actual = test_data.groupby('customer_id')['article_id'].apply(list).to_dict()

print(f"Период Train: {train_data['t_dat'].min().date()} - {train_data['t_dat'].max().date()} ({len(train_data)} строк)")
print(f"Период Val:   {val_data['t_dat'].min().date()} - {val_data['t_dat'].max().date()} ({len(val_data)} строк)")
print(f"Период Test:  {test_data['t_dat'].min().date()} - {test_data['t_dat'].max().date()} ({len(test_data)} строк)")
print(f"Всего уникальных клиентов в Val: {len(val_actual)}")
print(f"Всего уникальных клиентов в Test: {len(test_actual)}")

Период Train: 2020-08-23 - 2020-09-07 (570808 строк)
Период Val:   2020-09-08 - 2020-09-14 (241773 строк)
Период Test:  2020-09-15 - 2020-09-22 (241515 строк)
Всего уникальных клиентов в Val: 74575
Всего уникальных клиентов в Test: 75481


Bаseline 1. Рекомендация топ 20 самых популярных товаров

In [ ]:
top_20 = train_data['article_id'].value_counts().head(20).index.tolist()

print(f"Топ-20 популярных товаров: {top_20}")

Топ-20 популярных товаров: [751471001, 915526001, 915529003, 751471043, 706016001, 918292001, 898694001, 863595006, 915526002, 916468003, 915529001, 896152002, 863583002, 714790020, 884319001, 865929003, 914441004, 933706001, 803757001, 874754002]


In [ ]:
def calculate_metrics(actual_dict, predicted_list, k=12):
    precisions = []
    recalls = []
    
    for customer_id, actual_items in actual_dict.items():
        actual_set = set(actual_items)
        predicted_set = set(predicted_list[:k])
        
        hits = len(actual_set & predicted_set)
        
        precisions.append(hits / k)
        
        recalls.append(hits / len(actual_set) if len(actual_set) > 0 else 0)
        
    return np.mean(precisions), np.mean(recalls)

p_val, r_val = calculate_metrics(val_actual, top_20)

p_test, r_test = calculate_metrics(test_actual, top_20)

print(f"Валидация (Val): Precision@20 = {p_val:.5f}, Recall@20 = {r_val:.5f}")
print(f"Тест (Test):      Precision@20 = {p_test:.5f}, Recall@20 = {r_test:.5f}")

Валидация (Val): Precision@20 = 0.00502, Recall@20 = 0.02278
Тест (Test):      Precision@20 = 0.00413, Recall@20 = 0.01858


Bаseline 2. Рекомендация топ 20 самых популярных товаров а возрастной группе

In [ ]:
age_tops = {}
for group in train_data['age_group'].unique():
    if pd.isna(group): continue
    age_tops[group] = train_data[train_data['age_group'] == group]['article_id'].value_counts().head(20).index.tolist()

user_to_group = dict(zip(customers['customer_id'], customers['age_group']))

In [ ]:
def calculate_metrics_age(actual_dict, age_tops, user_groups_dict, global_top, k=20):
    precisions = []
    recalls = []
    
    for customer_id, actual_items in actual_dict.items():
        actual_set = set(actual_items)
        
        group = user_groups_dict.get(customer_id)
        
        predicted_list = age_tops.get(group, global_top)
        predicted_set = set(predicted_list[:k])
        
        hits = len(actual_set & predicted_set)
        
        precisions.append(hits / k)
        recalls.append(hits / len(actual_set) if len(actual_set) > 0 else 0)
        
    return np.mean(precisions), np.mean(recalls)

p_val_age, r_val_age = calculate_metrics_age(val_actual, age_tops, user_to_group, top_20)
p_test_age, r_test_age = calculate_metrics_age(test_actual, age_tops, user_to_group, top_20)

print(f"Валидация (Val) Age-Pop: Precision@20 = {p_val_age:.5f}, Recall@20 = {r_val_age:.5f}")
print(f"Тест (Test) Age-Pop: Precision@20 = {p_test_age:.5f}, Recall@20 = {r_test_age:.5f}")

Валидация (Val) Age-Pop: Precision@20 = 0.00488, Recall@20 = 0.03673
Тест (Test) Age-Pop: Precision@20 = 0.00418, Recall@20 = 0.03079


In [ ]:
metrics_dict = {
    "global_popularity": {
        "val_precision_20": float(p_val),
        "val_recall_20": float(r_val),
        "test_precision_20": float(p_test),
        "test_recall_20": float(r_test)
    },
    "age_based_popularity": {
        "val_precision_20": float(p_val_age),
        "val_recall_20": float(r_val_age),
        "test_precision_20": float(p_test_age),
        "test_recall_20": float(r_test_age)
    }
}

with open('../artifacts/baseline_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=4)

In [ ]:
with open('../artifacts/top_20.json', 'w') as f:
    json.dump(top_20_global, f, indent=4)

clean_age_tops = {str(k): v for k, v in age_group_tops.items()}

with open('../artifacts/age_group_tops_20.json', 'w') as f:
    json.dump(clean_age_tops, f, indent=4)